In [2]:
import duckdb

In [3]:
con = duckdb.connect()
con.execute(open("/workspace/sql/00_attach.sql").read())
print(con.sql("FROM ducklake_snapshots('lake')").df())

    snapshot_id                    snapshot_time  schema_version  \
0             0 2026-06-25 19:47:58.592626+00:00               0   
1             1 2026-06-25 19:47:58.638004+00:00               1   
2             2 2026-06-25 19:47:58.647187+00:00               2   
3             3 2026-06-25 19:47:58.669349+00:00               3   
4             4 2026-06-25 22:18:44.460503+00:00               4   
5             5 2026-06-25 23:13:50.427815+00:00               5   
6             6 2026-06-25 23:13:50.502705+00:00               6   
7             7 2026-06-26 00:31:41.238716+00:00               7   
8             8 2026-06-26 00:31:56.698735+00:00               8   
9             9 2026-06-26 00:32:06.396628+00:00               9   
10           10 2026-06-26 01:47:33.156378+00:00              10   
11           11 2026-06-26 01:47:40.993846+00:00              11   
12           12 2026-06-26 01:57:16.493581+00:00              12   
13           13 2026-06-26 01:57:16.512439+00:00

In [4]:
# read silver.coco_annotations as it was at snapshot 7 (first time it was created)
con.sql("""
    SELECT COUNT(*) AS row_count, 'snapshot 7' AS label
    FROM silver.coco_annotations AT (VERSION => 7)
""").df()

,row_count,label
0,36781,snapshot 7


In [5]:
# adjust timestamp to one between snapshots 7 and the drop/recreate
con.sql("""
    SELECT COUNT(*) AS row_count
    FROM silver.coco_annotations AT (TIMESTAMP => TIMESTAMPTZ '2026-06-26 00:32:00+00')
""").df()

,row_count
0,36781


In [9]:
snapshots = con.sql("FROM ducklake_snapshots('lake')").df()
print(snapshots[["snapshot_id", "changes"]])

    snapshot_id                                            changes
0             0                      {'schemas_created': ['main']}
1             1                       {'schemas_created': ['raw']}
2             2                    {'schemas_created': ['silver']}
3             3                      {'schemas_created': ['gold']}
4             4  {'tables_created': ['raw.coco_annotations'], '...
5             5  {'tables_created': ['raw.visdrone_fragments'],...
6             6  {'tables_created': ['raw.visdrone_annotations'...
7             7  {'tables_created': ['silver.coco_annotations']...
8             8  {'tables_created': ['silver.visdrone_annotatio...
9             9  {'tables_created': ['silver.visdrone_fragments...
10           10  {'tables_created': ['gold.coco_training'], 'ta...
11           11  {'tables_created': ['gold.visdrone_training'],...
12           12                          {'tables_dropped': ['7']}
13           13                         {'tables_dropped': ['1

In [12]:
first_silver = 7   # snapshot where silver.coco_annotations was first created
last = int(snapshots.iloc[-1]["snapshot_id"])
print(f"Comparing snapshot {first_silver} to {last}")
con.sql(f"""
    SELECT * FROM ducklake_table_changes('lake', 'silver', 'coco_annotations', {first_silver}, {last})
""").df()

Comparing snapshot 7 to 15


,snapshot_id,rowid,change_type,image_uri,image_id,width,height,bbox_id,category,bbox_x,bbox_y,bbox_w,bbox_h,area,split
0,14,0,insert,s3://lakehouse/assets/coco/images/139.jpg,139,640,426,2204286,dining table,321.21,231.22,446.77,320.15,2362.48975,val
1,14,1,insert,s3://lakehouse/assets/coco/images/632.jpg,632,640,483,1988858,book,527.02,248.57,551.42,289.00,245.04670,val
2,14,2,insert,s3://lakehouse/assets/coco/images/872.jpg,872,621,640,633972,baseball glove,368.64,157.25,426.09,203.03,1332.65760,val
3,14,3,insert,s3://lakehouse/assets/coco/images/1000.jpg,1000,640,480,1259139,person,265.33,95.86,354.25,411.74,14372.61775,val
4,14,4,insert,s3://lakehouse/assets/coco/images/1000.jpg,1000,640,480,1755199,person,52.14,185.12,111.40,397.65,6349.54530,val
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
36776,14,36776,insert,s3://lakehouse/assets/coco/images/580410.jpg,580410,427,640,1659515,book,323.34,191.37,327.30,211.32,69.84625,val
36777,14,36777,insert,s3://lakehouse/assets/coco/images/581206.jpg,581206,479,640,458653,person,0.00,2.53,469.53,217.79,53359.63985,val
36778,14,36778,insert,s3://lakehouse/assets/coco/images/581317.jpg,581317,640,354,327762,cell phone,406.82,142.23,428.25,178.48,311.76175,val
36779,14,36779,insert,s3://lakehouse/assets/coco/images/581357.jpg,581357,612,612,1691804,person,180.89,440.51,201.99,478.87,371.49395,val


In [13]:
# corrupt gold.coco_training to demonstrate rollback
last_good = int(con.sql("FROM ducklake_snapshots('lake')").df().iloc[-1]["snapshot_id"])
print(f"Last good snapshot before corruption: {last_good}")

con.execute("DROP TABLE gold.coco_training")
con.execute("""
    CREATE TABLE gold.coco_training AS
    SELECT image_uri, 'CORRUPTED' AS category, 0.0 AS bbox_x, 0.0 AS bbox_y,
           0.0 AS bbox_w, 0.0 AS bbox_h, split
    FROM silver.coco_annotations
""")
print("Bad transform applied.")
con.sql("SELECT category, COUNT(*) FROM gold.coco_training GROUP BY category").df()


Last good snapshot before corruption: 15
Bad transform applied.


,category,count_star()
0,CORRUPTED,36781


In [14]:
print(con.sql("FROM ducklake_snapshots('lake')").df().tail(3))

    snapshot_id                    snapshot_time  schema_version  \
15           15 2026-06-26 02:10:32.633864+00:00              15   
16           16 2026-06-26 03:07:59.123683+00:00              16   
17           17 2026-06-26 03:07:59.158607+00:00              17   

                                              changes author commit_message  \
15  {'tables_created': ['gold.coco_training'], 'ta...   None           None   
16                         {'tables_dropped': ['13']}   None           None   
17  {'tables_created': ['gold.coco_training'], 'ta...   None           None   

   commit_extra_info  
15              None  
16              None  
17              None  


In [15]:
con.execute("DROP TABLE gold.coco_training")
con.execute(f"""
    CREATE TABLE gold.coco_training AS
    SELECT * FROM gold.coco_training AT (VERSION => {last_good})
""")
print("Rolled back.")
con.sql("SELECT category, COUNT(*) FROM gold.coco_training GROUP BY category LIMIT 5").df()

Rolled back.


,category,count_star()
0,book,1161
1,boat,430
2,frisbee,115
3,bus,285
4,bed,163


In [16]:
print(con.sql("FROM ducklake_snapshots('lake')").df().tail(3))

    snapshot_id                    snapshot_time  schema_version  \
17           17 2026-06-26 03:07:59.158607+00:00              17   
18           18 2026-06-26 03:08:27.281367+00:00              18   
19           19 2026-06-26 03:08:27.305056+00:00              19   

                                              changes author commit_message  \
17  {'tables_created': ['gold.coco_training'], 'ta...   None           None   
18                         {'tables_dropped': ['14']}   None           None   
19  {'tables_created': ['gold.coco_training'], 'ta...   None           None   

   commit_extra_info  
17              None  
18              None  
19              None  


In [17]:
con.close()